### Ingestão da tabela de candidatos (consulta_cand)

Este notebook demonstra a ingestão da tabela específica de candidatos do TSE a partir dos arquivos CSV presentes em data/, com padronização de tipos de colunas e gravação em formatos Parquet e Delta para facilitar a análise posterior.


#### 1) Importações de bibliotecas para a ingestão da tabela votação



Neste bloco são carregadas as bibliotecas essenciais para o processo de ingestão. Elas permitem iniciar o Spark, habilitar o suporte ao Delta Lake e realizar operações básicas de manipulação e visualização dos dados.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline


##### 2) Inicializando a sessão Spark para carregar a tabela de votação

Este bloco cria a sessão Spark com as configurações necessárias para trabalhar com arquivos Delta e preparar o ambiente da pipeline de ingestão. É o ponto de partida para ler, transformar e gravar os dados corretamente.

In [2]:
builder = (
    SparkSession.builder
                .appName("TSE-Analytics-Validation")
                .config("spark.sql.extensions","io.delta.sql.DeltaSparkSessionExtension")
                .config("spark.sql.catalog.spark_catalog","org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Sessão Spark criada com sucesso com suporte a Delta Lake!")
print(f"Versão do PySpark: {spark.version}")


:: loading settings :: url = jar:file:/usr/local/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8e1b31d7-ab5b-4c33-ab23-1742d0eb21f9;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.3.1 in central
	found io.delta#delta-storage;4.3.1 in central
	found io.unitycatalog#unitycatalog-client;0.5.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.5.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.3.1 in central
	foun

Sessão Spark criada com sucesso com suporte a Delta Lake!
Versão do PySpark: 4.1.1


##### 3) Leitura da tabela específica de candidatos (Brasil) para ingestão

Neste bloco, os arquivos CSV da tabela de votação são lidos com o Spark a partir dos dados armazenados em data/. A leitura é feita com os parâmetros de codificação e separador corretos, permitindo que a informação seja preparada para os próximos passos da ingestão.

In [3]:
anos = list(range(2000,2025,2))

for i in anos:

    df = (spark.read
               .format("csv")
               .option("header", "true")
               .option("sep", ";")
               .option("encoding", "ISO-8859-1")
               .option("inferSchema", "true")
               .load(f"data/{i}/votacao_candidato_munzona_*/*_BRASIL.csv"))
    
    # Padronização de tipos de colunas para evitar conflitos de tipos no Parquet e erros de conversão de esquema
    if "SQ_CANDIDATO" in df.columns:
        df = df.withColumn("SQ_CANDIDATO", F.col("SQ_CANDIDATO").cast("bigint"))
    if "SQ_COLIGACAO" in df.columns:
        df = df.withColumn("SQ_COLIGACAO", F.col("SQ_COLIGACAO").cast("bigint"))
    if "SG_UE" in df.columns:
        df = df.withColumn("SG_UE", F.col("SG_UE").cast("string"))
    if "VR_DESPESA_MAX_CAMPANHA" in df.columns:
        df = df.withColumn("VR_DESPESA_MAX_CAMPANHA", F.col("VR_DESPESA_MAX_CAMPANHA").cast("double"))

    df.coalesce(1).write.format("parquet").mode("overwrite").save(f"data/raw/votacao/{i}")

26/07/21 14:31:47 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/2000/votacao_candidato_munzona_*/*_BRASIL.csv.
java.io.FileNotFoundException: File data/2000/votacao_candidato_munzona_*/*_BRASIL.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource

In [4]:
df = (spark.read
           .format("parquet")
           .option("inferSchema", "true")
           .load("data/raw/votacao/*"))

(df.coalesce(1)
   .write
   .format("delta")
   .mode("overwrite")
   .save("data/bronze/votacao"))

26/07/21 14:51:32 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/raw/votacao/*.
java.io.FileNotFoundException: File data/raw/votacao/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveDa